# 🧠 Intelligent Query Router with LangGraph

In this notebook, we build an **Intelligent Query Router** using **LangGraph**.
The router analyzes user queries, classifies their intent, and routes them to specialized nodes (e.g., Code Expert, Web Search / Knowledge Retrieval, General Assistant).

## 1. Setup & Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify key presence (without revealing values)
google_key = os.getenv("GOOGLE_API_KEY")
print(f"Google API Key configured: {bool(google_key)}")

## 2. Define State Schema & Helpers

We define the state structure and a helper function to extract clean plain text from responses.

In [ ]:
from typing import Annotated, Literal, TypedDict
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# Define Router Output Structure
class RouteQuery(BaseModel):
    """Route query to the most appropriate handler."""
    destination: Literal["code_expert", "web_search", "general_assistant"] = Field(
        description="Given a user query, choose the most relevant destination node."
    )
    reasoning: str = Field(description="Brief explanation for the routing decision.")

# Define Graph State
class RouterState(TypedDict):
    messages: Annotated[list, add_messages]
    destination: str
    query: str
    response: str

def extract_text(content) -> str:
    """Extract clean readable text and strip metadata/signatures to prevent horizontal scroll."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                text_parts.append(item.get("text", ""))
            elif isinstance(item, str):
                text_parts.append(item)
        return "\n".join(text_parts)
    return str(content)

## 3. Initialize Model & Router

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Using gemini-3.5-flash-lite for fast, stable, and high-throughput routing
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_retries=3)

structured_router = llm.with_structured_output(RouteQuery)

def router_node(state: RouterState):
    """Classifies query and decides the routing path."""
    query = state["query"]
    prompt = f"""Analyze the following user query and decide where to route it:
- 'code_expert': Queries regarding programming, algorithms, debugging, code review, or technical architectures.
- 'web_search': Queries about recent events, current facts, weather, news, or external up-to-date lookups.
- 'general_assistant': Conversational queries, creative writing, brainstorming, or general questions.

User query: {query}"""
    
    result = structured_router.invoke(prompt)
    print(f"\n[Router Decision] -> Destination: {result.destination}\nReason: {result.reasoning}")
    return {"destination": result.destination}

## 4. Define Specialized Handler Nodes

In [ ]:
def code_expert_node(state: RouterState):
    """Handles programming and technical queries."""
    query = state["query"]
    prompt = f"You are an expert software engineer. Provide a clean, robust, and well-explained solution for:\n\n{query}"
    response = llm.invoke(prompt)
    return {"response": extract_text(response.content)}

def web_search_node(state: RouterState):
    """Handles queries requiring search or fresh information."""
    query = state["query"]
    prompt = f"You are a search analyst. Provide accurate, factual information regarding:\n\n{query}"
    response = llm.invoke(prompt)
    return {"response": extract_text(response.content)}

def general_assistant_node(state: RouterState):
    """Handles general conversational queries."""
    query = state["query"]
    prompt = f"You are a helpful and friendly AI assistant. Answer the following query:\n\n{query}"
    response = llm.invoke(prompt)
    return {"response": extract_text(response.content)}

## 5. Build and Compile the LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END

# Initialize graph with state schema
builder = StateGraph(RouterState)

# Add nodes
builder.add_node("router", router_node)
builder.add_node("code_expert", code_expert_node)
builder.add_node("web_search", web_search_node)
builder.add_node("general_assistant", general_assistant_node)

# Connect entry point
builder.add_edge(START, "router")

# Conditional routing function
def route_decision(state: RouterState) -> str:
    return state["destination"]

# Add conditional edges from router node
builder.add_conditional_edges(
    "router",
    route_decision,
    {
        "code_expert": "code_expert",
        "web_search": "web_search",
        "general_assistant": "general_assistant"
    }
)

# Connect handler nodes to END
builder.add_edge("code_expert", END)
builder.add_edge("web_search", END)
builder.add_edge("general_assistant", END)

# Compile graph
graph = builder.compile()

## 6. Visualize the Graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Mermaid visualization requires pygraphviz / internet access. Graph compiled successfully!")

## 7. Test Queries (Formatted with Auto-Wrapping)

In [23]:
import time
from IPython.display import Markdown, display

test_queries = [
    "Write a Python function to reverse a linked list in O(n) time.",
    "What is the latest score of the ongoing Premier League match today?",
    "Can you write a poem about the morning sunrise over the mountains?"
]

for q in test_queries:
    print("\n" + "="*60)
    print(f"📌 Query: {q}")
    result = graph.invoke({"query": q})
    print(f"🎯 Routed Node: {result.get('destination')}")
    print("💬 Response:")
    # display Markdown to enable auto-wrapping and eliminate horizontal scrollbars
    display(Markdown(result["response"]))
    time.sleep(1)  # 1-second pause to prevent rate-limit spikes



📌 Query: Write a Python function to reverse a linked list in O(n) time.


d:\LangraphProjects\02_Intelligent_Query_Router\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Router Decision] -> Destination: code_expert
Reason: The query asks for a Python function implementing an algorithm (reversing a linked list), which falls under the scope of programming and technical topics handled by the code_expert.


d:\LangraphProjects\02_Intelligent_Query_Router\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


🎯 Routed Node: code_expert
💬 Response:


...

d:\LangraphProjects\02_Intelligent_Query_Router\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Router Decision] -> Destination: general_assistant
Reason: The user is asking for a creative poem, which falls under creative writing and general assistant capabilities rather than coding or real-time web search.


d:\LangraphProjects\02_Intelligent_Query_Router\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


🎯 Routed Node: general_assistant
💬 Response:


Here is a poem celebrating the beauty of a mountain sunrise:

**Awakening on the Peaks**

The night retreats in quiet grace,
As shadows leave the mountain's face.
The jagged peaks, so tall and grand,
Stand silent guard across the land.

Then comes a whisper in the sky,
Where soft and violet clouds drift by.
A blush of pink, a streak of gold,
As dawn's sweet story starts to unfold.

The sun breaks through the eastern height,
And bathes the slopes in liquid light.
It kisses snow, it wakes the pines,
And paints the crests in amber lines.

The valley stirs, the shadows fade,
In every quiet glen and glade.
A hopeful start, a world reborn,
Beneath the majesty of morn.

***

I hope you enjoyed the poem! Let me know if you'd like a poem about anything else.